# Dimensional Modelling

The normalised database works perfectly for day-to-day operations. Booking a visit, updating a patient's address, registering a new doctor -- all clean, all consistent.

But the analytics team needs something different. They want to slice A&E performance by time, department, patient demographics, and admission type. They want to answer questions like "what is the average wait time on Bank Holidays?" and "which triage categories have the longest waits for ambulance arrivals in Q3?"

Normalised schemas are not built for this. The many JOINs that keep data clean for OLTP make analytical queries slow and awkward. This is where **dimensional modelling** comes in -- a different way of structuring data, optimised for analysis rather than operations.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/nhs_trust.sqlite

## OLTP vs OLAP

Two fundamentally different workloads, two fundamentally different designs.

| | OLTP (Online Transaction Processing) | OLAP (Online Analytical Processing) |
|---|---|---|
| **Purpose** | Run the business | Analyse the business |
| **Users** | Clinicians, receptionists, administrators | Analysts, managers, board members |
| **Queries** | Insert one visit, update one address | Aggregate wait times across 6 months |
| **Optimised for** | Write speed, data integrity | Read speed, query flexibility |
| **Schema** | Normalised (3NF) | Denormalised (star/snowflake) |
| **Row count per query** | One or a few | Thousands to millions |

Our normalised database is OLTP. It is excellent at its job. But asking it analytical questions means writing multi-table JOINs for every query, and the database engine has to traverse many foreign key relationships to assemble each answer.

An OLAP schema pre-arranges the data for analysis. It deliberately introduces *controlled* redundancy (the exact thing normalisation eliminates) because the trade-off is worth it: faster queries, simpler SQL, and analytical dimensions that do not exist in the operational system at all.

## The star schema

The star schema is the most common OLAP design pattern. The name comes from its shape:

```
              dim_patient
                  |
dim_date --- fact_ae_visits --- dim_admission_type
                  |
            dim_department
```

At the centre is a **fact table** containing the measurements you want to analyse -- the numbers. Surrounding it are **dimension tables** containing the attributes you want to slice and filter by.

**Fact tables** contain:
- Foreign keys to each dimension
- Numeric measures (wait time, treatment time, cost)
- One row per event

**Dimension tables** contain:
- A primary key
- Descriptive attributes (patient name, date details, department name)
- Often denormalised -- a dimension table might include fields that in OLTP would live in separate tables

The key insight: every analytical query follows the same pattern. Pick measures from the fact table, filter and group by attributes from the dimensions. One JOIN per dimension, always the same shape.

## Building our star schema

Let's build a star schema for A&E performance analysis. We will create the dimension tables first, then the fact table.

### dim_date: the date dimension

This is the dimension that earns its keep most dramatically.

In the raw data, arrival times are stored as timestamps: `2024-03-15 14:32`. That is great for recording when something happened. It is terrible for answering questions like:

- What is the average wait time on **Bank Holidays**?
- How does performance differ between **weekdays and weekends**?
- What is the trend by **quarter**?

A date dimension pre-computes all these attributes for every date in your range. Want to know if 15th March 2024 is a Bank Holiday? Just look it up.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS dim_date (
    date_key TEXT PRIMARY KEY,           -- '2024-01-15'
    full_date TEXT NOT NULL,
    day_of_week INTEGER NOT NULL,        -- 0=Monday, 6=Sunday
    day_name TEXT NOT NULL,              -- 'Monday'
    day_of_month INTEGER NOT NULL,       -- 1-31
    week_of_year INTEGER NOT NULL,       -- 1-53
    month INTEGER NOT NULL,              -- 1-12
    month_name TEXT NOT NULL,            -- 'January'
    quarter INTEGER NOT NULL,            -- 1-4
    year INTEGER NOT NULL,
    is_weekend INTEGER NOT NULL,         -- 0 or 1
    is_bank_holiday INTEGER NOT NULL,    -- 0 or 1
    bank_holiday_name TEXT               -- NULL or 'Christmas Day' etc.
);

Now let's populate it. We need every date in 2024 (the range of our A&E data), with Bank Holidays marked.

In [ ]:
from datetime import datetime, timedelta
import sqlite3

# England Bank Holidays 2024
bank_holidays = {
    "2024-01-01": "New Year's Day",
    "2024-03-29": "Good Friday",
    "2024-04-01": "Easter Monday",
    "2024-05-06": "Early May Bank Holiday",
    "2024-05-27": "Spring Bank Holiday",
    "2024-08-26": "Summer Bank Holiday",
    "2024-12-25": "Christmas Day",
    "2024-12-26": "Boxing Day",
}

day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
month_names = ["", "January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

conn = sqlite3.connect("../data/nhs_trust.sqlite")
cur = conn.cursor()

# Clear any existing data
cur.execute("DELETE FROM dim_date")

start = datetime(2024, 1, 1)
end = datetime(2024, 12, 31)
current = start

rows_inserted = 0
while current <= end:
    date_str = current.strftime("%Y-%m-%d")
    dow = current.weekday()  # 0=Monday
    is_weekend = 1 if dow >= 5 else 0
    is_bh = 1 if date_str in bank_holidays else 0
    bh_name = bank_holidays.get(date_str)
    quarter = (current.month - 1) // 3 + 1
    week = current.isocalendar()[1]

    cur.execute(
        "INSERT INTO dim_date VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (date_str, date_str, dow, day_names[dow], current.day, week,
         current.month, month_names[current.month], quarter, current.year,
         is_weekend, is_bh, bh_name)
    )
    rows_inserted += 1
    current += timedelta(days=1)

conn.commit()
conn.close()

print(f"Inserted {rows_inserted} dates into dim_date")

Let's check that our Bank Holidays are in there:

In [ ]:
%%sql
SELECT date_key, day_name, bank_holiday_name
FROM dim_date
WHERE is_bank_holiday = 1
ORDER BY date_key;

### dim_patient

The patient dimension captures the attributes we want to analyse by: age group, location, GP practice. In a star schema, we often denormalise -- the GP practice name goes directly into the patient dimension rather than being a separate lookup.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS dim_patient (
    patient_key INTEGER PRIMARY KEY,
    nhs_number TEXT NOT NULL,
    name TEXT NOT NULL,
    dob TEXT NOT NULL,
    postcode TEXT NOT NULL,
    postcode_area TEXT NOT NULL,       -- first part of postcode for regional analysis
    gp_practice_name TEXT NOT NULL
);

In [ ]:
%%sql
INSERT OR IGNORE INTO dim_patient (patient_key, nhs_number, name, dob, postcode, postcode_area, gp_practice_name)
SELECT
    p.patient_id,
    p.nhs_number,
    p.name,
    p.dob,
    p.postcode,
    SUBSTR(p.postcode, 1, INSTR(p.postcode, ' ') - 1) AS postcode_area,
    gp.name
FROM patients p
JOIN gp_practices gp ON p.gp_practice_id = gp.gp_practice_id;

In [ ]:
%sql SELECT * FROM dim_patient LIMIT 5;

Notice that `gp_practice_name` is stored directly in the patient dimension. In OLTP this would be a normalisation violation. In OLAP it is standard practice -- the analyst does not want to join through three tables just to filter by GP practice.

### dim_department

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS dim_department (
    department_key INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    floor INTEGER NOT NULL
);

In [ ]:
%%sql
INSERT OR IGNORE INTO dim_department (department_key, name, floor)
SELECT department_id, name, floor
FROM departments;

### dim_admission_type

This is a small dimension -- sometimes called a "junk dimension" when it captures miscellaneous categorical attributes. Here it is clean: just the types of A&E admission.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS dim_admission_type (
    admission_type_key INTEGER PRIMARY KEY AUTOINCREMENT,
    admission_type TEXT NOT NULL UNIQUE
);

In [ ]:
%%sql
INSERT OR IGNORE INTO dim_admission_type (admission_type)
SELECT DISTINCT admission_type
FROM ae_visits
ORDER BY admission_type;

In [ ]:
%sql SELECT * FROM dim_admission_type;

### fact_ae_visits: the fact table

The fact table is where the measures live. Each row is one A&E visit, and the foreign keys point to each dimension.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS fact_ae_visits (
    ae_fact_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key TEXT NOT NULL,
    patient_key INTEGER NOT NULL,
    department_key INTEGER NOT NULL,
    admission_type_key INTEGER NOT NULL,
    -- Measures
    triage_category INTEGER NOT NULL,
    wait_minutes INTEGER NOT NULL,
    treatment_minutes INTEGER NOT NULL,
    total_minutes INTEGER NOT NULL,
    outcome TEXT NOT NULL,
    arrival_hour INTEGER NOT NULL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (patient_key) REFERENCES dim_patient(patient_key),
    FOREIGN KEY (department_key) REFERENCES dim_department(department_key),
    FOREIGN KEY (admission_type_key) REFERENCES dim_admission_type(admission_type_key)
);

In [ ]:
%%sql
INSERT OR IGNORE INTO fact_ae_visits (
    date_key, patient_key, department_key, admission_type_key,
    triage_category, wait_minutes, treatment_minutes, total_minutes,
    outcome, arrival_hour
)
SELECT
    DATE(ae.arrival_time) AS date_key,
    ae.patient_id AS patient_key,
    ae.department_id AS department_key,
    at2.admission_type_key,
    ae.triage_category,
    ae.wait_minutes,
    ae.treatment_minutes,
    ae.wait_minutes + ae.treatment_minutes AS total_minutes,
    ae.outcome,
    CAST(SUBSTR(ae.arrival_time, 12, 2) AS INTEGER) AS arrival_hour
FROM ae_visits ae
JOIN dim_admission_type at2 ON ae.admission_type = at2.admission_type;

In [ ]:
%sql SELECT COUNT(*) AS total_facts FROM fact_ae_visits;

In [ ]:
%sql SELECT * FROM fact_ae_visits LIMIT 5;

## The payoff: analytical queries

Now watch how naturally analytical questions map to SQL against the star schema.

### Average wait time on Bank Holidays vs normal days

This is the cognitive anchor. Try answering this question with just raw timestamps and no date dimension. You would need to hard-code every Bank Holiday date into your WHERE clause, or build a lookup on the fly. With `dim_date`, it is one JOIN and one filter.

In [ ]:
%%sql
SELECT
    CASE WHEN d.is_bank_holiday = 1 THEN 'Bank Holiday'
         WHEN d.is_weekend = 1 THEN 'Weekend'
         ELSE 'Weekday'
    END AS day_type,
    COUNT(*) AS visits,
    ROUND(AVG(f.wait_minutes), 1) AS avg_wait_mins,
    ROUND(AVG(f.treatment_minutes), 1) AS avg_treatment_mins
FROM fact_ae_visits f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY day_type
ORDER BY avg_wait_mins DESC;

One query. No hard-coded dates. If Parliament declares a new Bank Holiday, you add one row to `dim_date` and every query that references Bank Holidays automatically includes it.

### Wait times by triage category and admission type

In [ ]:
%%sql
SELECT
    at2.admission_type,
    f.triage_category,
    COUNT(*) AS visits,
    ROUND(AVG(f.wait_minutes), 1) AS avg_wait_mins
FROM fact_ae_visits f
JOIN dim_admission_type at2 ON f.admission_type_key = at2.admission_type_key
GROUP BY at2.admission_type, f.triage_category
ORDER BY at2.admission_type, f.triage_category;

### Monthly trends

In [ ]:
%%sql
SELECT
    d.month_name,
    d.month,
    COUNT(*) AS visits,
    ROUND(AVG(f.wait_minutes), 1) AS avg_wait_mins,
    ROUND(AVG(f.total_minutes), 1) AS avg_total_mins
FROM fact_ae_visits f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.month_name, d.month
ORDER BY d.month;

### Outcomes by quarter

What proportion of A&E visits result in admission, by quarter?

In [ ]:
%%sql
SELECT
    'Q' || d.quarter AS quarter,
    f.outcome,
    COUNT(*) AS visits,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.quarter), 1) AS pct
FROM fact_ae_visits f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.quarter, f.outcome
ORDER BY d.quarter, visits DESC;

Notice the pattern. Every query has the same shape:

1. SELECT measures from the fact table
2. JOIN to the dimensions you need
3. GROUP BY dimension attributes

This predictability is a feature, not a limitation. It means analysts can learn the pattern once and apply it to any question. It means BI tools can auto-generate queries. It means new analysts on the team can be productive quickly.

## Slowly Changing Dimensions (SCDs)

There is a question we have been avoiding: what happens when dimension data changes?

A patient moves house. Their postcode changes from CB1 to PE2. In the normalised OLTP database, you simply update their row. But in the analytical warehouse, you have a choice to make.

### Type 1: Overwrite

Just update the dimension row. The old value is gone.

```sql
UPDATE dim_patient
SET postcode = 'PE2 7HH', postcode_area = 'PE2'
WHERE patient_key = 42;
```

**Pros:** Simple. Dimension table stays small.

**Cons:** Historical analysis breaks. If you query "how many A&E visits came from CB1 patients?", visits that happened when the patient lived in CB1 now show PE2. History has been rewritten.

### Type 2: Keep history

Create a new row for the patient with the new details, and mark the old row as expired.

```sql
-- Expire the old row
UPDATE dim_patient
SET effective_end = '2024-06-30', is_current = 0
WHERE patient_key = 42;

-- Insert a new row with new details
INSERT INTO dim_patient (patient_key, nhs_number, name, dob, postcode,
                         postcode_area, gp_practice_name, effective_start,
                         effective_end, is_current)
VALUES (4200, '1234567890', 'Jane Smith', '1985-03-12', 'PE2 7HH',
        'PE2', 'Ashford Family Practice', '2024-07-01', '9999-12-31', 1);
```

**Pros:** Complete history. Visits that happened when the patient was in CB1 still link to the CB1 version of the row.

**Cons:** More complex. The dimension table grows. Queries need to be aware of `is_current` and date ranges.

### Which to choose?

It depends on the question you are trying to answer.

- If you need to know where a patient lives **now** (for operational reporting), Type 1 is fine.
- If you need to know where a patient lived **at the time of their visit** (for epidemiological analysis, catchment area studies), you need Type 2.

In NHS settings, Type 2 is common for patient demographics because understanding historical geographical patterns matters for public health planning.

## Star schema vs normalised schema: a direct comparison

Let's answer the same question against both schemas and compare.

**Question:** What is the average wait time by day of week?

Against the OLTP schema (no date dimension):

In [ ]:
%%sql
-- OLTP approach: extract day of week from raw timestamp
SELECT
    CASE CAST(STRFTIME('%w', arrival_time) AS INTEGER)
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END AS day_name,
    ROUND(AVG(wait_minutes), 1) AS avg_wait
FROM ae_visits
GROUP BY STRFTIME('%w', arrival_time)
ORDER BY STRFTIME('%w', arrival_time);

Against the star schema:

In [ ]:
%%sql
-- OLAP approach: join to date dimension
SELECT
    d.day_name,
    ROUND(AVG(f.wait_minutes), 1) AS avg_wait
FROM fact_ae_visits f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.day_name, d.day_of_week
ORDER BY d.day_of_week;

Same answer, but the OLAP version is cleaner. And the moment you want to add "but only on Bank Holidays" or "in Q3 only", the star schema query barely changes -- you just add a WHERE clause. The OLTP version needs increasingly baroque CASE expressions and date arithmetic.

## Summary

| Concept | What it means |
|---|---|
| **OLTP** | Operational systems: normalised, optimised for writes and integrity |
| **OLAP** | Analytical systems: denormalised, optimised for reads and aggregation |
| **Star schema** | Fact table at the centre, dimension tables around it |
| **Fact table** | Contains measures (numbers) and foreign keys to dimensions |
| **Dimension table** | Contains descriptive attributes for slicing and filtering |
| **Date dimension** | Pre-computed calendar attributes: weekday, quarter, Bank Holiday, etc. |
| **SCD Type 1** | Overwrite old values -- loses history |
| **SCD Type 2** | Keep old rows with date ranges -- preserves history |

The normalised schema and the star schema serve different purposes. Neither is better in the abstract -- they answer different questions for different users. A well-run data platform has both: OLTP for operations, OLAP for analysis, and ETL pipelines to move data between them.

---

## Exercises

### Exercise 1: Peak hours analysis

Using the star schema, write a query to find the **average wait time by arrival hour** (0-23). Which hours have the longest waits? Does this make intuitive sense for an A&E department?

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 2: Bank Holiday deep dive

Write a query to show the **average wait time for each specific Bank Holiday** (using `bank_holiday_name` from `dim_date`). Which Bank Holiday has the worst performance?

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 3: Design an outpatient star schema

The trust wants a separate analytical schema for **outpatient clinic** performance using the `visits` table.

Design the star schema on paper (or in comments) first, then write the SQL to create:

1. `fact_outpatient_visits` -- what measures would you include?
2. What dimension tables do you need?
3. Populate the fact table from the OLTP `visits` table.

Think about what questions the analytics team might ask about outpatient clinics.

In [ ]:
%%sql

-- YOUR SQL HERE
-- Design and build a star schema for outpatient clinic analysis


### Exercise 4: SCD Type 2 simulation

Patient 42 has moved from their current address to a new postcode: `NR3 4XY`. The move happened on 2024-07-15.

Write the SQL statements that a Type 2 SCD process would execute. You will need to:

1. First, check the patient's current details in `dim_patient`
2. Think about what additional columns you would need on `dim_patient` to support Type 2 (effective dates, current flag)
3. Write the UPDATE to expire the old row and the INSERT for the new row

You do not need to actually alter the table -- just write the SQL as comments or in a markdown cell explaining your approach.

In [ ]:
%%sql

-- First, look up patient 42's current details
SELECT * FROM dim_patient WHERE patient_key = 42;

In [ ]:
-- YOUR SQL / COMMENTS HERE
-- Write the Type 2 SCD statements
